# cuQuantum gradient reproduction

Run with a CUDA-enabled JAX kernel containing jaxquantum from `feature/cqt-jax`, cuQuantum, and the private cuQuantum JAX PID 0.0.7. The PID must be installed separately; this notebook does not include it. The examples require no local files or repository-relative imports.

This notebook compares gradients through `jaxquantum.sesolve`, then reduces the mismatch to two direct cuDensityMat `operator_action` calls.

In [1]:
import sys
from importlib.metadata import PackageNotFoundError, version

import jax
jax.config.update("jax_enable_x64", True)  # before importing cuQuantum
import jax.numpy as jnp
import cuquantum
import jaxquantum as jqt
from cuquantum.densitymat.jax import (
    ElementaryOperator, Operator, OperatorTerm, State, operator_action,
)

assert any(device.platform == "gpu" for device in jax.devices())
print("JAX:", jax.__version__, "cuQuantum:", cuquantum.__version__)
try:
    print("cuQuantum JAX PID:", version("cuquantum-python-jax-cu13"))
except PackageNotFoundError:
    print("cuQuantum JAX PID version unavailable from package metadata")
print("Devices:", jax.devices())

JAX: 0.8.3 cuQuantum: 26.6.0
cuQuantum JAX PID: 0.0.7
Devices: [CudaDevice(id=0)]


## 1. Gradient through sesolve

For H(theta) = theta * sigma_x / 2, initial ket |0>, and final time t = 0.2, the |1> population is sin(theta * t / 2)^2. Its exact derivative is t * sin(theta * t) / 2. Forward results should match for both backends.

In [2]:
theta = jnp.array(0.7, dtype=jnp.float64)
t_final = 0.2
times = jnp.array([0.0, t_final])
ket0 = jqt.basis(2, 0)
options = jqt.SolverOptions(progress_meter=None)

def population(x, backend):
    hamiltonian = (x / 2) * jqt.sigmax(implementation=backend)
    ket = jqt.sesolve(hamiltonian, ket0, times, solver_options=options).data[-1]
    return jnp.real(jnp.abs(ket[1]) ** 2)

eps = 1e-4
exact_grad = float(t_final * jnp.sin(theta * t_final) / 2)
print(f"Analytic gradient: {exact_grad:.9f}")
print(f"{'backend':<12} {'population':>12} {'jax.grad':>12} {'finite diff':>12}")
results = {}
for backend in ("dense", "cuquantum"):
    f = lambda x: population(x, backend)
    value = float(f(theta))
    gradient = float(jax.grad(f)(theta))
    finite_difference = float((f(theta + eps) - f(theta - eps)) / (2 * eps))
    results[backend] = (value, gradient, finite_difference)
    print(f"{backend:<12} {value:12.9f} {gradient:12.9f} {finite_difference:12.9f}")

assert abs(results["dense"][0] - results["cuquantum"][0]) < 1e-8
assert abs(results["dense"][1] - exact_grad) < 1e-6

Analytic gradient: 0.013954311
backend        population     jax.grad  finite diff


dense         0.004892002  0.013954311  0.013954311


cuquantum     0.004892002  0.063834368  0.013954311


## 2. Minimal reproduction: two operator_action calls

This cell uses cuDensityMat directly, with no jaxquantum solver or Diffrax. Let A = -i * theta * sigma_x / 2. The real part of the first component of A^2|0> is -theta^2/4, so its exact gradient at theta = 0.7 is -0.35. The versions below differ only in whether both actions reuse one Operator.

In [3]:
sigma_x = jnp.array([[0, 1], [1, 0]], dtype=jnp.complex128)
state0 = jnp.array([1, 0], dtype=jnp.complex128)

def make_operator(x):
    term = OperatorTerm((2,))
    term.append([ElementaryOperator(sigma_x)], modes=(0,), coeff=x / 2)
    op = Operator((2,), simplify=True)
    op.append(term, dual=False, coeff=-1j)
    return op

def apply(op, state):
    return operator_action(op, State(state)).get_data()[0]

def two_actions(x, reuse_operator):
    first_op = make_operator(x)
    after_first = apply(first_op, state0)
    second_op = first_op if reuse_operator else make_operator(x)
    after_second = apply(second_op, after_first)
    return jnp.real(after_second[0])

print(f"Exact value: {-float(theta**2)/4:.9f}; exact gradient: {-float(theta)/2:.9f}")
print(f"{'operator use':<20} {'value':>12} {'jax.grad':>12}")
for label, reuse in (("fresh each action", False), ("reuse same operator", True)):
    f = lambda x: two_actions(x, reuse)
    print(f"{label:<20} {float(f(theta)):12.9f} {float(jax.grad(f)(theta)):12.9f}")

Exact value: -0.122500000; exact gradient: -0.350000000
operator use                value     jax.grad


fresh each action    -0.122500000 -0.350000000
reuse same operator  -0.122500000 -0.525000000


**Interpretation.** Reusing one Operator reproduces the wrong gradient. The installed PID's gradient callbacks accumulate into persistent buffers; they appear not to reset between backward actions. Replacing accumulation with assignment is not a general fix because one action may need to sum several contributions. Use the dense backend for gradients through time evolution until this is fixed upstream. If a newer PID produces matching gradients, the issue may have been resolved.